# 08 · Measure retrieval quality — the baseline

> **Run order.** This notebook is step 8 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


The number everything after this is measured against.

**Zero LLM calls.** Recall@k and MRR compare retrieved element IDs against the
ground-truth element IDs notebook 06 anchored — pure computation. That matters
practically as well as intellectually: the most iterative days of this project
cost nothing against any provider's rate limit.

A hit means a retrieved chunk contains one of the expected `element_id`s.
`pR@5` is the softer *page*-level recall: did we surface the right page at all?

In [1]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

import time
from analyst.benchmark import BenchmarkQuestion
from analyst.config import get_settings
from analyst.embedding import Embedder
from analyst.vectorstore import VectorStore

K_VALUES = (1, 3, 5, 10)
MAX_K = max(K_VALUES)
settings = get_settings()

path = settings.data_dir / "benchmark" / "questions.jsonl"
questions = [BenchmarkQuestion.model_validate_json(line)
             for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"{len(questions)} benchmark questions")
pd.DataFrame([q.model_dump() for q in questions]).groupby(
    ["question_type", "match_kind"]).size().to_frame("n")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


44 benchmark questions

n
question_type match_kind    
growth        exact        9
              tolerant     1
value_lookup  exact       25
              tolerant     9

In [2]:
def evaluate(model: str, use_filters: bool = True):
    embedder = Embedder(model)
    store = VectorStore(settings.qdrant_url,
                        f"{settings.collection_prefix}_{model}", embedder.dim)
    per_q = []
    for q in questions:
        expected = set(q.expected_element_ids)
        expected_pages = set(q.expected_pages)
        t0 = time.perf_counter()
        hits = store.search(
            embedder.embed_query(q.question), limit=MAX_K,
            ticker=q.ticker if use_filters else None,
            # A growth question spans two years, so a single-year filter would
            # exclude half its evidence by construction.
            fiscal_year=(q.fiscal_year
                         if use_filters and q.question_type == "value_lookup" else None),
        )
        ms = (time.perf_counter() - t0) * 1000
        rank = next((i for i, h in enumerate(hits, 1) if expected & set(h.element_ids)), None)
        prank = next((i for i, h in enumerate(hits, 1) if expected_pages & set(h.pages)), None)
        per_q.append({"question_type": q.question_type, "ticker": q.ticker,
                      "rank": rank, "page_rank": prank, "ms": ms})
    return pd.DataFrame(per_q)

def summarise(df: pd.DataFrame, label: str) -> dict:
    row = {"config": label, "n": len(df)}
    for k in K_VALUES:
        row[f"R@{k}"] = round((df["rank"].notna() & (df["rank"] <= k)).mean(), 3)
    row["MRR"] = round(df["rank"].apply(lambda r: 1 / r if pd.notna(r) else 0.0).mean(), 3)
    row["pR@5"] = round((df["page_rank"].notna() & (df["page_rank"] <= 5)).mean(), 3)
    row["p50_ms"] = round(df["ms"].median(), 1)
    return row

print("evaluator ready")

evaluator ready

## Baseline: dense retrieval, metadata filters on

In [3]:
base = evaluate("bge-small", use_filters=True)
pd.DataFrame([summarise(base, "bge-small + filters")])

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


,config,n,R@1,R@3,R@5,R@10,MRR,pR@5,p50_ms
0,bge-small + filters,44,0.023,0.045,0.045,0.091,0.039,0.091,74.1


## Does metadata filtering actually earn its complexity?

Filtering restricts the candidate set *before* scoring. The claim is that it is both faster and more accurate than filtering afterwards. Claims get measured here, not asserted.

In [4]:
nofilter = evaluate("bge-small", use_filters=False)
pd.DataFrame([
    summarise(nofilter, "bge-small, no filters"),
    summarise(base, "bge-small + filters"),
])

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


,config,n,R@1,R@3,R@5,R@10,MRR,pR@5,p50_ms
0,"bge-small, no filters",44,0.023,0.045,0.045,0.091,0.035,0.091,73.8
1,bge-small + filters,44,0.023,0.045,0.045,0.091,0.039,0.091,74.1


## Where does it fail?

Aggregate numbers hide the interesting part.

In [5]:
by_type = base.groupby("question_type").apply(
    lambda g: pd.Series(summarise(g, g.name)), include_groups=False)
print(by_type.to_string())

print("\nQuestions where the correct element was NEVER retrieved in the top 10:")
misses = base[base["rank"].isna()]
print(f"  {len(misses)} of {len(base)}")
misses.groupby(["ticker", "question_type"]).size().to_frame("misses")

                     config   n    R@1    R@3    R@5   R@10    MRR   pR@5  p50_ms
question_type                                                                    
growth               growth  10  0.000  0.000  0.000  0.000  0.000  0.100    82.0
value_lookup   value_lookup  34  0.029  0.059  0.059  0.118  0.051  0.088    73.3


Questions where the correct element was NEVER retrieved in the top 10:

  40 of 44

misses
ticker    question_type        
HDFCBANK  value_lookup        2
ICICIBANK growth              3
          value_lookup        6
RELIANCE  value_lookup        5
SUNPHARMA growth              7
          value_lookup       17

## Baseline recorded

Day 4 adds sparse (lexical) retrieval and a cross-encoder reranker, and re-runs exactly this notebook. The comparison is the point: *\"reranking took Recall@5 from X to Y\"* is worth ten projects that say \"I used a reranker\".